In [1]:
!pip install -U openai-whisper
!pip install librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=f8a8447159e2a24db239ac35bf46bd523e5dc05bceb1aecf3321f7ee7654bea7
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [3]:
import whisper
import librosa
import numpy as np
import re
import torch

# ============================================================
# Hyperparams (tempo / filler / scaling은 네 설정 유지)
# ============================================================
TEMPO_MU = 125
TEMPO_SIGMA = 40
ALPHA = 0.03

PITCH_LOW, PITCH_HIGH = 30, 65
BETA = 0.04

E_LOW, E_HIGH = 0.02, 0.05
GAMMA_ENERGY = 30.0

K_FILLER = 5

# emphasis detection (그대로 유지)
PITCH_Z_TH_STRONG = 1.5
ENERGY_Z_TH_STRONG = 1.0
PITCH_Z_TH_WEAK = 1.2
ENERGY_Z_TH_WEAK = 0.8
MIN_EMPH_SEC = 0.20

# emphasis score band (그대로 유지)
ER_LOW = 2.0
ER_HIGH = 8.0
DELTA_EMPH = 0.08

# Feature extraction alignment
SR = 16000
HOP = 512
FRAME_LENGTH = 2048

# ============================================================
# Personalization controls (NEW)
# ============================================================
# calibration length: T_cal = clip(0.15*T_total, 5, 10)
CAL_FRAC = 0.15
CAL_MIN_SEC = 5.0
CAL_MAX_SEC = 10.0

# micro pause ignore
MICRO_PAUSE_SEC = 0.30

# realtime / scoring thresholds in z-units (personal baseline)
Z0_PITCH = 2.0     # scoring band
Z0_ENERGY = 2.0

Z_HIGH = 2.2       # realtime alert band
Z_LOW  = 2.2

# score decay strengths (튜닝 포인트)
LAMBDA_P = 3.0     # pitch violation ratio penalty
LAMBDA_E = 3.0     # energy violation ratio penalty
LAMBDA_S = 3.0     # pause excess penalty

# pause threshold from personal baseline: tau = mu + k*sigma
K_PAUSE = 2.0
FALLBACK_TAU_PAUSE = 1.0  # cal구간에 pause가 거의 없을 때

# ============================================================
# Total / Scaling controls (네 설정 유지)
# ============================================================
LAM_FLUENCY = 0.55
LAM_TEMPO   = 0.20

FLOOR_TEMPO = 20.0

CALIB_P = 0.55
CALIB_MIN = 0.0
CALIB_MAX = 75.0

USE_MIN_MIX = False
MIN_MIX_W = 0.40

# ============================================================
# Models
# ============================================================
model = whisper.load_model("small")
audio_files = ["/content/test1.wav", "/content/test2.wav", "/content/test3.wav"]

# Silero VAD (Colab에서 internet OK)
# (torch.hub 기반 로드)
vad_model, vad_utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils

# ============================================================
# Utils
# ============================================================
def clip01_to_100(x):
    return float(np.clip(x, 0.0, 100.0))

def compute_wpm(text, speech_duration_sec):
    words = len(text.split())
    if speech_duration_sec <= 1e-9:
        return 0.0
    return words / (speech_duration_sec / 60.0)

def count_fillers_korean(text, fillers):
    tokens = text.lower().split()
    counts = {f: 0 for f in fillers}
    for t in tokens:
        t_clean = re.sub(r"[^\w가-힣]", "", t)
        if t_clean in counts:
            counts[t_clean] += 1
    return counts

def compute_pitch_energy(y, sr):
    # frame-aligned pitch + rms
    f0, _, _ = librosa.pyin(y, fmin=70, fmax=400, sr=sr, hop_length=HOP)
    pitch_mean = float(np.nanmean(f0)) if np.any(~np.isnan(f0)) else float("nan")
    pitch_std  = float(np.nanstd(f0))  if np.any(~np.isnan(f0)) else float("nan")

    rms = librosa.feature.rms(y=y, frame_length=FRAME_LENGTH, hop_length=HOP)[0]
    energy_mean = float(np.mean(rms))
    energy_std  = float(np.std(rms))
    return pitch_mean, pitch_std, energy_mean, energy_std, f0, rms

def intervals_to_frame_mask(intervals_sec, n_frames, sr=SR, hop=HOP):
    # frame midpoint time
    frame_times = (np.arange(n_frames) * hop) / sr
    mask = np.zeros(n_frames, dtype=bool)
    for (s, e) in intervals_sec:
        mask |= (frame_times >= s) & (frame_times <= e)
    return mask, frame_times

def vad_speech_intervals(y, sr):
    # silero expects torch tensor 1D float32
    wav = torch.from_numpy(y.astype(np.float32))
    ts = get_speech_timestamps(wav, vad_model, sampling_rate=sr)
    # convert samples->sec intervals
    intervals = []
    for t in ts:
        intervals.append((t["start"]/sr, t["end"]/sr))
    return intervals

def pause_gaps_from_speech(intervals):
    # intervals sorted
    if len(intervals) < 2:
        return []
    gaps = []
    for i in range(len(intervals)-1):
        gap = max(0.0, intervals[i+1][0] - intervals[i][1])
        if gap >= MICRO_PAUSE_SEC:
            gaps.append(gap)
    return gaps

def clip_calibration_length(T_total):
    return float(np.clip(CAL_FRAC * T_total, CAL_MIN_SEC, CAL_MAX_SEC))

# ============================================================
# Original tempo/fluency scores (네 수식 유지)
# ============================================================
def score_rate_wpm(wpm, mu=TEMPO_MU, sigma=TEMPO_SIGMA):
    s = 100.0 * (1.0 - ((wpm - mu) ** 2) / (sigma ** 2))
    return clip01_to_100(s)

def segment_wpm_variance(whisper_result):
    seg_wpms = []
    for seg in whisper_result.get("segments", []):
        start = float(seg.get("start", 0.0))
        end   = float(seg.get("end", 0.0))
        dur = max(1e-6, end - start)
        words = len(str(seg.get("text", "")).strip().split())
        if words == 0:
            continue
        seg_wpms.append(words / (dur / 60.0))
    if len(seg_wpms) < 2:
        return 0.0
    return float(np.std(seg_wpms))

def score_stability(var_wpm, alpha=ALPHA):
    return clip01_to_100(100.0 * np.exp(-alpha * var_wpm))

def score_tempo(s_rate, s_stability):
    return clip01_to_100(0.6 * s_rate + 0.4 * s_stability)

def score_fluency_filler(text, filler_counts, k=K_FILLER):
    n_words = max(1, len(text.split()))
    n_filler = sum(filler_counts.values())
    fr = n_filler / n_words
    s = 100.0 * (1.0 - k * fr)
    return clip01_to_100(s), float(fr)

# ============================================================
# Emphasis (네 것 유지)
# ============================================================
def _zscore(x):
    mu = np.mean(x)
    sd = np.std(x) + 1e-8
    return (x - mu) / sd

def _count_runs(mask, min_frames):
    n = 0
    run = 0
    for v in mask:
        if v:
            run += 1
        else:
            if run >= min_frames:
                n += 1
            run = 0
    if run >= min_frames:
        n += 1
    return n

def count_emphasis_weighted(f0, rms, sr, speech_mask=None):
    frame_sec = HOP / sr
    L = min(len(f0), len(rms))
    f0 = f0[:L]
    rms = rms[:L]

    if speech_mask is None:
        speech_mask = np.ones(L, dtype=bool)
    else:
        speech_mask = speech_mask[:L]

    voiced = (~np.isnan(f0)) & speech_mask
    if np.sum(voiced) < 8:
        return 0.0, 0.0, 0, 0

    f0_z = np.full(L, -np.inf, dtype=float)
    f0_z[voiced] = _zscore(f0[voiced])
    rms_z = np.full(L, -np.inf, dtype=float)
    rms_z[speech_mask] = _zscore(rms[speech_mask])

    strong_mask = (f0_z > PITCH_Z_TH_STRONG) & (rms_z > ENERGY_Z_TH_STRONG)
    weak_mask   = (f0_z > PITCH_Z_TH_WEAK) | (rms_z > ENERGY_Z_TH_WEAK)

    min_frames = int(np.ceil(MIN_EMPH_SEC / frame_sec))
    strong_cnt = _count_runs(strong_mask, min_frames)
    weak_cnt   = _count_runs(weak_mask, min_frames)

    extra_weak = max(0, weak_cnt - strong_cnt)
    weighted = 1.0 * strong_cnt + 0.5 * extra_weak

    duration_sec = L * frame_sec
    er_per_min = weighted / max(1e-6, duration_sec / 60.0)
    return float(weighted), float(er_per_min), int(strong_cnt), int(extra_weak)

def score_emphasis_band(er_per_min, er_low=ER_LOW, er_high=ER_HIGH, delta=DELTA_EMPH):
    if er_per_min <= 0:
        return 0.0
    if er_per_min < er_low:
        return clip01_to_100(100.0 * (er_per_min / er_low))
    if er_low <= er_per_min <= er_high:
        return 100.0
    return clip01_to_100(100.0 * np.exp(-delta * (er_per_min - er_high)))

# ============================================================
# NEW: Personal baseline + scoring for pitch/energy/pause (VAD 기반)
# ============================================================
def personal_baseline(f0, rms, frame_times, speech_mask, T_cal):
    cal_mask = (frame_times <= T_cal) & speech_mask

    # pitch baseline (voiced only)
    voiced_cal = cal_mask & (~np.isnan(f0))
    if np.sum(voiced_cal) >= 8:
        mu_f0 = float(np.mean(f0[voiced_cal]))
        sd_f0 = float(np.std(f0[voiced_cal]) + 1e-8)
    else:
        mu_f0, sd_f0 = float(np.nanmean(f0)), float(np.nanstd(f0) + 1e-8)

    # energy baseline (speech only)
    if np.sum(cal_mask) >= 8:
        mu_e = float(np.mean(rms[cal_mask]))
        sd_e = float(np.std(rms[cal_mask]) + 1e-8)
    else:
        mu_e, sd_e = float(np.mean(rms)), float(np.std(rms) + 1e-8)

    return mu_f0, sd_f0, mu_e, sd_e

def pitch_energy_scores_personal(f0, rms, frame_times, speech_mask, T_cal, T_total,
                                mu_f0, sd_f0, mu_e, sd_e):
    eval_mask = (frame_times > T_cal) & (frame_times <= T_total) & speech_mask

    # pitch violation ratio over eval speech frames
    voiced_eval = eval_mask & (~np.isnan(f0))
    if np.sum(voiced_eval) > 0:
        zf0 = (f0[voiced_eval] - mu_f0) / sd_f0
        r_pitch = float(np.mean(np.abs(zf0) > Z0_PITCH))
    else:
        r_pitch = 0.0

    # energy violation ratio
    if np.sum(eval_mask) > 0:
        ze = (rms[eval_mask] - mu_e) / sd_e
        r_energy = float(np.mean(np.abs(ze) > Z0_ENERGY))
    else:
        r_energy = 0.0

    S_pitch = 100.0 * np.exp(-LAMBDA_P * r_pitch)
    S_energy = 100.0 * np.exp(-LAMBDA_E * r_energy)
    return clip01_to_100(S_pitch), clip01_to_100(S_energy), r_pitch, r_energy

def pause_score_personal(speech_intervals, T_cal, T_total):
    # split intervals into cal/eval by cutting at T_cal
    # For pause baseline: gaps whose both sides are within [0, T_cal]
    cal_intervals = [(s,e) for (s,e) in speech_intervals if e <= T_cal]
    eval_intervals = [(s,e) for (s,e) in speech_intervals if e > T_cal]

    cal_gaps = pause_gaps_from_speech(cal_intervals)
    if len(cal_gaps) >= 1:
        mu_pause = float(np.mean(cal_gaps))
        sd_pause = float(np.std(cal_gaps) + 1e-8)
        tau = mu_pause + K_PAUSE * sd_pause
    else:
        mu_pause, sd_pause = 0.0, 0.0
        tau = FALLBACK_TAU_PAUSE

    # eval gaps: between segments in eval area (approx)
    eval_gaps = pause_gaps_from_speech(eval_intervals)
    # total excess pause seconds normalized by eval duration
    T_eval = max(1e-6, (T_total - T_cal))
    excess = 0.0
    for g in eval_gaps:
        excess += max(0.0, g - tau)
    r_pause = float(excess / T_eval)

    S_pause = 100.0 * np.exp(-LAMBDA_S * r_pause)
    return clip01_to_100(S_pause), r_pause, tau, mu_pause, sd_pause, eval_gaps

# ============================================================
# Total + Calibration (네 방식 유지, core만 개인점수로 대체)
# ============================================================
def compute_total_raw(scores):
    tempo = max(FLOOR_TEMPO, scores["tempo"])
    base_components = [tempo, scores["pitch"], scores["energy"], scores["pause"]]
    base = float(np.mean(base_components))

    mult_f = (1.0 - LAM_FLUENCY * (1.0 - scores["fluency"]/100.0))
    mult_t = (1.0 - LAM_TEMPO   * (1.0 - tempo/100.0))
    mult_f = np.clip(mult_f, 0.0, 1.0)
    mult_t = np.clip(mult_t, 0.0, 1.0)

    gated = base * mult_f * mult_t

    if USE_MIN_MIX:
        min_core = float(np.min(base_components + [scores["fluency"]]))
        total_raw = (1.0 - MIN_MIX_W) * gated + MIN_MIX_W * min_core
        return float(total_raw), float(base), float(gated), float(min_core), float(mult_f), float(mult_t)

    return float(gated), float(base), float(gated), float(np.min(base_components)), float(mult_f), float(mult_t)

def calibrate_to_100(raw, p=CALIB_P, lo=CALIB_MIN, hi=CALIB_MAX):
    x = (raw - lo) / max(1e-9, (hi - lo))
    x = np.clip(x, 0.0, 1.0)
    return float(100.0 * (x ** p))

# ============================================================
# "Realtime" log simulation on wav (chunk processing)
# ============================================================
def realtime_log_simulation(f0, rms, frame_times, speech_mask, T_cal,
                            mu_f0, sd_f0, mu_e, sd_e, tau_pause,
                            speech_intervals):
    """
    실제 마이크 스트리밍 대신, wav를 프레임 순서대로 돌면서
    '지금 너무 높다/낮다' 경고를 timestamp와 함께 찍는 시뮬레이션.
    """
    print("---- Realtime alerts (simulated) ----")
    # pitch/energy alerts: only after calibration + within speech
    for i, t in enumerate(frame_times):
        if t <= T_cal:
            continue
        if not speech_mask[i]:
            continue

        # pitch alert (voiced)
        if not np.isnan(f0[i]):
            z = (f0[i] - mu_f0) / sd_f0
            if z > Z_HIGH:
                print(f"[{t:6.2f}s] pitch HIGH (z={z:.2f})")
            elif z < -Z_LOW:
                print(f"[{t:6.2f}s] pitch LOW  (z={z:.2f})")

        # energy alert
        zE = (rms[i] - mu_e) / sd_e
        if zE > Z_HIGH:
            print(f"[{t:6.2f}s] energy HIGH (z={zE:.2f})")
        elif zE < -Z_LOW:
            print(f"[{t:6.2f}s] energy LOW  (z={zE:.2f})")

    # pause alert: gap-based (VAD)
    # after calibration only
    eval_intervals = [(s,e) for (s,e) in speech_intervals if e > T_cal]
    gaps = pause_gaps_from_speech(eval_intervals)
    if len(eval_intervals) >= 2:
        for i in range(len(eval_intervals)-1):
            gap = max(0.0, eval_intervals[i+1][0] - eval_intervals[i][1])
            if gap >= MICRO_PAUSE_SEC and gap > tau_pause:
                print(f"[{eval_intervals[i][1]:6.2f}s] pause TOO LONG (gap={gap:.2f}s > tau={tau_pause:.2f}s)")
    print("-----------------------------------")

# ============================================================
# Run
# ============================================================
for audio_path in audio_files:
    print(f"\n=== {audio_path} ===")

    # audio load
    y, sr = librosa.load(audio_path, sr=SR)
    T_total = float(librosa.get_duration(y=y, sr=sr))
    T_cal = clip_calibration_length(T_total)
    T_eval = max(1e-6, T_total - T_cal)

    # VAD speech intervals (sec)
    speech_intervals = vad_speech_intervals(y, sr)
    T_speech = sum((e - s) for (s, e) in speech_intervals)

    # ASR
    result = model.transcribe(audio_path, language="ko")
    text = result["text"].strip()

    # pitch/energy frame features
    pitch_mean, pitch_std, energy_mean, energy_std, f0, rms = compute_pitch_energy(y, sr)
    n_frames = min(len(f0), len(rms))
    f0 = f0[:n_frames]
    rms = rms[:n_frames]

    speech_mask, frame_times = intervals_to_frame_mask(speech_intervals, n_frames, sr=sr, hop=HOP)

    # ===== personal baseline (0 ~ T_cal)
    mu_f0, sd_f0, mu_e, sd_e = personal_baseline(f0, rms, frame_times, speech_mask, T_cal)

    # ===== personal pitch/energy scores (T_cal ~ T_total, speech-only)
    s_pitch_personal, s_energy_personal, r_pitch, r_energy = pitch_energy_scores_personal(
        f0, rms, frame_times, speech_mask, T_cal, T_total, mu_f0, sd_f0, mu_e, sd_e
    )

    # ===== personal pause score (VAD gap-based)
    s_pause_personal, r_pause, tau_pause, mu_pause, sd_pause, eval_gaps = pause_score_personal(
        speech_intervals, T_cal, T_total
    )

    # fillers
    fillers = ["어", "음", "그"]
    filler_counts = count_fillers_korean(text, fillers)
    s_fluency, fr = score_fluency_filler(text, filler_counts)

    # tempo (WPM은 "speech time" 기준이 더 자연스러움)
    # (silence가 길면 wpm이 과도하게 낮아지는 문제 방지)
    wpm = compute_wpm(text, max(1e-6, T_speech))

    s_rate = score_rate_wpm(wpm)
    var_wpm = segment_wpm_variance(result)
    s_stability = score_stability(var_wpm)
    s_tempo = score_tempo(s_rate, s_stability)

    # emphasis: speech frames만 넣어서 계산
    emph_weighted, er, strong_cnt, weak_extra = count_emphasis_weighted(f0, rms, sr, speech_mask=speech_mask)
    s_emph = score_emphasis_band(er)

    scores = {
        "tempo": s_tempo,
        "pitch": s_pitch_personal,
        "energy": s_energy_personal,
        "fluency": s_fluency,
        "emphasis": s_emph,
        "pause": s_pause_personal,
    }

    total_raw, base, gated, min_core, mult_f, mult_t = compute_total_raw(scores)
    total_final = calibrate_to_100(total_raw)

    # ===== Print summary
    print(f"T_total={T_total:.2f}s | T_cal={T_cal:.2f}s | T_eval={T_eval:.2f}s")
    print(f"VAD: speech_segments={len(speech_intervals)} | T_speech={T_speech:.2f}s")
    print(f"Text: {text}")
    print(f"WPM(speech-time): {wpm:.2f} | Var_WPM(segment): {var_wpm:.2f}")
    print(f"Pitch raw: mean={pitch_mean:.2f}Hz std={pitch_std:.2f}Hz  | personal μ={mu_f0:.2f} σ={sd_f0:.2f}")
    print(f"Energy raw: mean={energy_mean:.4f} std={energy_std:.4f}     | personal μ={mu_e:.4f} σ={sd_e:.4f}")
    print(f"Pitch violation r_pitch={r_pitch:.3f} | Energy violation r_energy={r_energy:.3f}")
    print(f"Pause: tau={tau_pause:.2f}s | r_pause(excess/T_eval)={r_pause:.3f} | eval_gaps={eval_gaps[:5]}")
    print(f"Filler Counts: {filler_counts} | FR: {fr:.4f}")
    print(f"Emphasis: weighted={emph_weighted:.2f} | ER(per min): {er:.2f} | strong={strong_cnt} | weak(extra)={weak_extra}")
    print("---- Scores (0~100) ----")
    print(f"Tempo Score: {s_tempo:.0f} (rate={s_rate:.0f}, stability={s_stability:.0f})")
    print(f"Pitch Score (personal): {scores['pitch']:.0f}")
    print(f"Energy Score (personal): {scores['energy']:.0f}")
    print(f"Pause Score (personal): {scores['pause']:.0f}")
    print(f"Fluency Score: {scores['fluency']:.0f}")
    print(f"Emphasis Score: {scores['emphasis']:.0f}")
    print("---- Total ----")
    print(f"Raw total: {total_raw:.1f} | Base: {base:.1f} | mult_f={mult_f:.2f}, mult_t={mult_t:.2f}")
    print(f"Total (calibrated 0~100): {total_final:.1f}")

    # ===== Realtime alert simulation (wav로 흉내)
    realtime_log_simulation(
        f0, rms, frame_times, speech_mask, T_cal,
        mu_f0, sd_f0, mu_e, sd_e, tau_pause,
        speech_intervals
    )

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip

=== /content/test1.wav ===
T_total=49.43s | T_cal=7.41s | T_eval=42.01s
VAD: speech_segments=13 | T_speech=42.89s
Text: 인생에 있어서 많은 것이 필요하다고는 생각지 않습니다. 튜브에 대사 인생이 뭐 별건가 달콤한 기억 하나면 그만이지처럼 고된 하루라도 소중하고 달콤한 기억 하나만 간직한다면 웃으며 살아가는 게 또 인생이라고 생각합니다. 그러나 그 말은 그 달콤한 기억, 짜릿한 성술을 위해 어떤 고통이라도 감내할 준비가 되어 있다는 말이기도 합니다. 지나간 과거에 연연하거나 다가오지 않은 미래를 두려워하지 않는 제가 되고 싶습니다. 소중한 하루를 밝게 웃으면서 살아가는 제가 되랍니다.
WPM(speech-time): 89.53 | Var_WPM(segment): 6.30
Pitch raw: mean=200.05Hz std=45.16Hz  | personal μ=215.86 σ=26.59
Energy raw: mean=0.0426 std=0.0259     | personal μ=0.0499 σ=0.0191
Pitch violation r_pitch=0.025 | Energy violation r_energy=0.117
Pause: tau=1.00s | r_pause(excess/T_eval)=0.005 | eval_gaps=[0.6759999999999984, 0.3240000000000016, 0.41999999999999815, 1.2199999999999989, 0.4200000000000017]
Filler Counts: {'어': 0, '음': 0, '그': 2} | FR: 0.0312
Emphasis: weighted=13.50 | ER(per min): 16.38 | 